### Trigger:
A trigger is database object that automatically runs sql code whenever there is update,insert and delete operations occurson on arun tables.


A **Trigger** in SQL is a **special stored program that automatically executes when a specific event happens** on a table or view.

Unlike stored procedures, **you do not call a trigger manually**. The database automatically fires it when an `INSERT`, `UPDATE`, or `DELETE` operation occurs.

> **Definition:** A trigger is a database object that automatically executes in response to DML (INSERT, UPDATE, DELETE) or sometimes DDL (CREATE, ALTER, DROP) events.

---

# Real-Life Analogy

Imagine an ATM.

When you withdraw money:

1. Money is deducted from your account.
2. Transaction history is updated.
3. SMS is sent.
4. Daily withdrawal limit is updated.

You don't manually perform steps 2, 3, and 4. They happen automatically.

A **trigger** works in the same way.

---

# How a Trigger Works

```text
User

↓

INSERT / UPDATE / DELETE

↓

Trigger Fires Automatically

↓

Business Logic Executes

↓

Transaction Completes
```

---

# Syntax (SQL Server)

```sql
CREATE TRIGGER TriggerName
ON TableName
AFTER INSERT
AS
BEGIN
    -- SQL Statements
END;
```

---

# Types of Triggers

There are mainly three DML trigger types.

| Trigger      | Executes When        |
| ------------ | -------------------- |
| AFTER INSERT | After inserting data |
| AFTER UPDATE | After updating data  |
| AFTER DELETE | After deleting data  |

Some databases also support:

* INSTEAD OF Trigger
* DDL Trigger
* LOGON Trigger (SQL Server)

---

# Example Database

## Employees Table

| EmpID | Name  | Salary |
| ----: | ----- | -----: |
|   101 | Rahul |  50000 |
|   102 | Priya |  60000 |

## Audit Table

| EmpID | Action | ActionDate |
| ----: | ------ | ---------- |
|       |        |            |

We'll use this table to record changes.

---

# 1. AFTER INSERT Trigger

Create trigger

```sql
CREATE TRIGGER trg_InsertEmployee
ON Employees
AFTER INSERT
AS
BEGIN
    INSERT INTO Audit(EmpID, Action, ActionDate)
    SELECT EmpID, 'Employee Added', GETDATE()
    FROM inserted;
END;
```

---

## What is the `inserted` table?

SQL Server automatically creates a temporary table named **`inserted`** containing the newly inserted rows.

Example:

```sql
INSERT INTO Employees
VALUES (103,'John',70000);
```

The `inserted` table contains:

| EmpID | Name | Salary |
| ----: | ---- | -----: |
|   103 | John |  70000 |

The trigger inserts into the audit table.

Audit table becomes:

| EmpID | Action         | ActionDate |
| ----: | -------------- | ---------- |
|   103 | Employee Added | 2026-07-23 |

The user only inserted into `Employees`; the audit row was created automatically.

---

# 2. AFTER UPDATE Trigger

Suppose salary changes need to be logged.

```sql
CREATE TRIGGER trg_UpdateSalary
ON Employees
AFTER UPDATE
AS
BEGIN
    INSERT INTO Audit(EmpID, Action, ActionDate)
    SELECT EmpID,
           'Salary Updated',
           GETDATE()
    FROM inserted;
END;
```

Update:

```sql
UPDATE Employees
SET Salary=80000
WHERE EmpID=101;
```

Audit table:

| EmpID | Action         |
| ----: | -------------- |
|   101 | Salary Updated |

---

## What are `inserted` and `deleted`?

During an update, SQL Server provides both:

**Before update (`deleted`)**

| EmpID | Salary |
| ----: | -----: |
|   101 |  50000 |

**After update (`inserted`)**

| EmpID | Salary |
| ----: | -----: |
|   101 |  80000 |

You can compare them.

Example:

```sql
SELECT
    d.Salary AS OldSalary,
    i.Salary AS NewSalary
FROM deleted d
JOIN inserted i
ON d.EmpID=i.EmpID;
```

Result:

| OldSalary | NewSalary |
| --------: | --------: |
|     50000 |     80000 |

---

# 3. AFTER DELETE Trigger

Create trigger

```sql
CREATE TRIGGER trg_DeleteEmployee
ON Employees
AFTER DELETE
AS
BEGIN
    INSERT INTO Audit(EmpID, Action, ActionDate)
    SELECT EmpID,
           'Employee Deleted',
           GETDATE()
    FROM deleted;
END;
```

Delete

```sql
DELETE
FROM Employees
WHERE EmpID=102;
```

The `deleted` table contains:

| EmpID | Name  |
| ----: | ----- |
|   102 | Priya |

Audit table:

| EmpID | Action           |
| ----: | ---------------- |
|   102 | Employee Deleted |

---

# 4. INSTEAD OF Trigger

Instead of executing the original statement, the trigger runs its own logic.

Example:

Prevent deletion.

```sql
CREATE TRIGGER trg_PreventDelete
ON Employees
INSTEAD OF DELETE
AS
BEGIN
    PRINT 'Deletion Not Allowed';
END;
```

Now

```sql
DELETE FROM Employees
WHERE EmpID=101;
```

Output

```
Deletion Not Allowed
```

The row is not deleted.

---

# Real-Time Examples

## 1. Banking System

Customer withdraws money.

```sql
UPDATE Accounts
SET Balance=Balance-500
WHERE AccountID=101;
```

Trigger automatically:

* Inserts transaction history
* Updates audit log
* Checks overdraft rules

---

## 2. E-Commerce

New order placed.

```sql
INSERT INTO Orders
VALUES(...)
```

Trigger automatically:

* Decreases product stock
* Creates invoice
* Writes order history

---

## 3. Hospital

Patient admitted.

```sql
INSERT INTO Patients
```

Trigger:

* Creates medical record
* Allocates room
* Generates patient ID

---

## 4. HR System

Employee salary updated.

```sql
UPDATE Employees
SET Salary=90000
```

Trigger:

* Stores old salary
* Records who updated it
* Records update time

---

## 5. Banking Audit

Deleting customer data.

```sql
DELETE FROM Customers
```

Trigger stores deleted data in an audit table for compliance.

---

# DDL Triggers

DDL triggers fire on schema changes.

Example:

```sql
CREATE TRIGGER trg_NoDropTable
ON DATABASE
FOR DROP_TABLE
AS
BEGIN
    PRINT 'Dropping Tables Not Allowed';
    ROLLBACK;
END;
```

Now

```sql
DROP TABLE Employees;
```

The operation is rolled back.

---

# Advantages

* Automatic execution
* Enforces business rules
* Keeps audit/history tables
* Improves data integrity
* Centralizes database logic
* No application code required for certain actions

---

# Disadvantages

### 1. Performance Overhead

Every insert, update, or delete also runs the trigger.

```
Insert Row

↓

Trigger Executes

↓

Insert Completes
```

More triggers mean more work.

---

### 2. Harder to Debug

The application may only execute:

```sql
INSERT INTO Employees
VALUES(...)
```

But multiple hidden triggers could also run, making behavior harder to trace.

---

### 3. Cascading Effects

One trigger can modify another table, which may fire another trigger, creating a chain of executions if not carefully designed.

---

# Trigger vs Stored Procedure

| Feature      | Trigger                            | Stored Procedure                       |
| ------------ | ---------------------------------- | -------------------------------------- |
| Execution    | Automatic                          | Manual (`EXEC`)                        |
| Triggered By | INSERT, UPDATE, DELETE, DDL events | User or application                    |
| Parameters   | No                                 | Yes                                    |
| Purpose      | Automatic business rules, auditing | Business operations and reusable logic |

Example:

Stored procedure:

```sql
EXEC AddEmployee;
```

Trigger:

```sql
INSERT INTO Employees
VALUES(...)
```

The trigger fires automatically—no explicit call is needed.

---

# Interview Example

Suppose your company wants to **track every salary change**.

**Without a trigger:**

* Every application that updates salaries must also insert a row into an audit table.
* If one application forgets, the audit trail is incomplete.

**With a trigger:**

* Any update to the `Employees` table—regardless of which application performs it—automatically records the change in the audit table.
* This ensures consistent auditing across all applications.

---

# Summary

| Trigger Type | Fires On                     | Common Real-Time Use                             |
| ------------ | ---------------------------- | ------------------------------------------------ |
| AFTER INSERT | New rows inserted            | Audit logs, welcome emails, stock updates        |
| AFTER UPDATE | Existing rows updated        | Salary history, balance tracking, change logging |
| AFTER DELETE | Rows deleted                 | Backup, audit, archive                           |
| INSTEAD OF   | Replaces the original action | Prevent deletes, validate complex operations     |
| DDL Trigger  | Schema changes               | Prevent dropping tables, log schema changes      |

## When are triggers used in real projects?

Triggers are commonly used for:

* **Audit logging** (record who changed what and when)
* **Maintaining history tables**
* **Enforcing business rules** at the database level
* **Archiving deleted records**
* **Synchronizing related tables** automatically

However, modern applications avoid putting all business logic in triggers because hidden logic can make systems harder to understand and maintain. They are best reserved for cross-application rules (such as auditing and data integrity) that should always be enforced, regardless of which application accesses the database.
